# Motivation: why physics beats data alone

*(Companion to Chapter 1, §1.3 of the book.)*

A neural network fitted to **data alone** is an interpolator: it reproduces the samples but
knows nothing of the law that generated them, so it behaves arbitrarily where data are absent.
A **physics-informed** network adds the governing equation as an *inductive bias* and recovers
the solution even where no data exist.

**The problem** — free vibration of a mass–spring–damper (underdamped harmonic oscillator):
$$\frac{d^2u}{dt^2} + 2\delta\,\frac{du}{dt} + \omega_0^2\,u = 0,\qquad u(0)=1,\ u'(0)=0,$$
with exact solution $u(t)=e^{-\delta t}\big(\cos\omega t + \tfrac{\delta}{\omega}\sin\omega t\big)$,
$\omega=\sqrt{\omega_0^2-\delta^2}$. We take $\delta=2$, $\omega_0=20$.

**The experiment** — we are given clean data in only the first 40% of the interval, and ask
each network to reproduce the *whole* interval. Verified result: the data-only network fails
in the no-data region (rel. $L_2 \approx 2$); the PINN matches the exact solution
(rel. $L_2 < 0.01$).

> Runs in ~1–2 minutes on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Problem, exact solution, and DATA in the first 40% only
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

delta, w0 = 2.0, 20.0
w = np.sqrt(w0**2 - delta**2)
def exact(t):
    return np.exp(-delta*t)*(np.cos(w*t) + (delta/w)*np.sin(w*t))

T = 1.0
td = np.linspace(0, 0.4, 20)          # DATA: first 40% of the domain only
ud = exact(td)                        # clean data
td_t = torch.tensor(td, dtype=torch.float32, device=device).reshape(-1, 1)
ud_t = torch.tensor(ud, dtype=torch.float32, device=device).reshape(-1, 1)

def make_net():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(),
                         nn.Linear(64,64), nn.Tanh(), nn.Linear(64,1)).to(device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

In [ ]:
# Cell 2 -- A purely DATA-DRIVEN network (no physics)
net_data = make_net()
opt = torch.optim.Adam(net_data.parameters(), 1e-3)
for e in range(8000):
    opt.zero_grad()
    ((net_data(td_t) - ud_t)**2).mean().backward()
    opt.step()
print('data-driven network trained')

In [ ]:
# Cell 3 -- A PHYSICS-INFORMED network: same data + the ODE residual over the WHOLE interval
# The residual is normalised by w0^2 so its terms are O(1) (good conditioning).
net_pinn = make_net()
opt = torch.optim.Adam(net_pinn.parameters(), 5e-3)
for e in range(20000):
    if e == 12000:
        for gp in opt.param_groups: gp['lr'] = 1e-3
    if e == 17000:
        for gp in opt.param_groups: gp['lr'] = 2e-4
    opt.zero_grad()
    t = (torch.rand(400, 1, device=device)*T).requires_grad_(True)   # collocation over [0,T]
    u = net_pinn(t); ut = g1(u, t); utt = g1(ut, t)
    res = utt/w0**2 + (2*delta/w0**2)*ut + u                          # normalised ODE residual
    loss = ((net_pinn(td_t) - ud_t)**2).mean() + 0.5*(res**2).mean()  # data + physics
    loss.backward(); opt.step()
    if e % 4000 == 0: print(f'epoch {e:5d}  loss {loss.item():.2e}')
if device.type == 'cuda': torch.cuda.synchronize()
print('physics-informed network trained')

In [ ]:
# Cell 4 -- Compare: both match the data; only the PINN recovers the no-data region
tg = np.linspace(0, T, 400)
tt = torch.tensor(tg, dtype=torch.float32, device=device).reshape(-1, 1)
ex = exact(tg)
with torch.no_grad():
    p_data = net_data(tt).cpu().numpy().ravel()
    p_pinn = net_pinn(tt).cpu().numpy().ravel()
rel = lambda p: np.sqrt(np.mean((p-ex)**2)/np.mean(ex**2))
print(f'relative L2 over [0,T]:  data-only = {rel(p_data):.3f}   PINN = {rel(p_pinn):.3f}')

plt.figure(figsize=(9, 4.3))
plt.axvspan(0, 0.4, color='gold', alpha=0.12)
plt.text(0.2, -1.08, 'data region', ha='center', color='0.4', fontsize=9)
plt.text(0.7, -1.08, 'no data (physics only)', ha='center', color='0.4', fontsize=9)
plt.plot(tg, ex, 'g', lw=3, alpha=0.55, label='exact solution')
plt.plot(tg, p_data, 'b--', lw=1.7, label=f'data-only NN (rel $L_2$={rel(p_data):.2f})')
plt.plot(tg, p_pinn, 'r-.', lw=1.7, label=f'PINN: data + physics (rel $L_2$={rel(p_pinn):.2f})')
plt.scatter(td, ud, c='k', s=18, zorder=5, label='data')
plt.axvline(0.4, color='0.6', ls=':', lw=1)
plt.xlabel('t'); plt.ylabel('u(t)'); plt.ylim(-1.25, 1.25); plt.grid(alpha=.3)
plt.legend(fontsize=9, loc='upper right')
plt.title('Physics lets the PINN recover the solution where there is no data')
plt.tight_layout(); plt.show()

## Takeaways

- **Data alone does not generalise.** The data-driven network reproduces the samples but, having
  no notion of the governing law, diverges to a meaningless trajectory outside the data region.
- **The residual is information.** The ODE, enforced at collocation points across the *whole*
  interval, constrains the network at infinitely many places where no data exist — turning an
  ill-posed extrapolation into a well-posed problem. This is the *inductive bias* a PINN adds.
- **Conditioning matters.** Normalising the residual by $\omega_0^2$ (so its terms are $O(1)$)
  is what lets the physics and data losses balance; without it the stiff $\omega_0^2 u$ term
  dominates and training stalls. Try removing the `/w0**2` factors and watch the match degrade.

**Experiments to try:** shrink the data region to the first 25%; add Gaussian noise to the data
and confirm the PINN still tracks the true solution (physics denoises); make $\delta$ or
$\omega_0$ trainable and recover it from the data (an inverse problem, Chapter 3).